[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sadriica/Curso_ANH/blob/main/modulo4_taller/modulo4_taller.ipynb)

Primera vez en Colab: ver la [guía](https://github.com/Sadriica/Curso_ANH/blob/main/guia_colab.md). Términos: [glosario](https://github.com/Sadriica/Curso_ANH/blob/main/glosario.md).

# Data & GIS para Energía
## Módulo 4: Aplicación. Reto. Taller.

El taller reúne los módulos anteriores en un flujo completo, con datos que llegan como en la vida
real: en distintos sistemas de coordenadas y en formatos distintos.

Objetivo: producir un mapa de idoneidad para un proyecto eólico en el norte de Colombia.

Pasos:

1. Cargar las fuentes (vienen en distintos CRS y formatos).
2. Unificar: llevar todo al mismo sistema de coordenadas.
3. Generar la malla (H3).
4. Mapear las fuentes sobre la malla.
5. Visualizar.
6. Decidir: combinar los criterios con AHP y obtener el mapa final.

Corre completo en Colab. Los archivos están también en `recursos/`.

## 0. Preparación

In [ ]:
!pip install -q geopandas "h3>=4.1" folium mapclassify

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import h3
import folium

## 1. Las fuentes (distintos CRS y formatos)

Dos fuentes de ejemplo:

- **Fuente A (recurso eólico y vías):** puntos en EPSG:9377 (metros, oficial de Colombia), con
  `viento_ms` y `dist_via_km`.
- **Fuente B (radiación solar):** una tabla en EPSG:4326 (grados lat/lon), con `radiacion`.

Aquí se generan; en un caso real se cargan con `read_file` o `read_csv`.

In [ ]:
base = pd.DataFrame({
    "sitio": ["Uribia", "Riohacha", "Maicao", "Manaure",
              "Valledupar", "Aguachica", "Santa Marta", "Barranquilla"],
    "lat": [11.71, 11.55, 11.38, 11.78, 10.46, 8.31, 11.24, 10.96],
    "lon": [-71.98, -72.91, -72.24, -72.44, -73.25, -73.63, -74.20, -74.80],
    "viento_ms":   [9.1, 7.3, 6.8, 8.7, 5.2, 4.1, 5.9, 6.2],
    "dist_via_km": [3.2, 0.5, 1.1, 4.8, 0.3, 0.9, 0.4, 0.2],
    "radiacion":   [6.1, 5.8, 5.9, 6.0, 5.2, 5.0, 5.4, 5.5],
})

# Fuente A: puntos en EPSG:9377 (metros)
fuente_a = gpd.GeoDataFrame(
    base[["sitio", "viento_ms", "dist_via_km"]],
    geometry=gpd.points_from_xy(base["lon"], base["lat"]), crs="EPSG:4326",
).to_crs("EPSG:9377")

# Fuente B: tabla en EPSG:4326 (grados)
fuente_b = base[["sitio", "lat", "lon", "radiacion"]].copy()

print("Fuente A CRS:", fuente_a.crs, "  (coordenadas en metros)")
print("Fuente B: tabla con lat/lon en grados (EPSG:4326)")
fuente_a.head(3)

## 2. Unificar: mismo sistema de coordenadas

La Fuente A está en metros y la B en grados: no se pueden cruzar directamente. Se lleva todo a
EPSG:4326.

In [ ]:
fuente_a_4326 = fuente_a.to_crs("EPSG:4326")

fuente_b_gdf = gpd.GeoDataFrame(
    fuente_b,
    geometry=gpd.points_from_xy(fuente_b["lon"], fuente_b["lat"]), crs="EPSG:4326",
)
print("A ahora en:", fuente_a_4326.crs, "| B en:", fuente_b_gdf.crs)

## 3. Generar la malla (H3)

Se crea una malla hexagonal que cubre la zona de estudio. Cada celda tiene un código único y
tamaño casi igual, lo que hace comparables los sitios.

In [ ]:
RES = 5
zona = {"type": "Polygon", "coordinates": [[
    [-75.2, 8.0], [-71.5, 8.0], [-71.5, 12.2], [-75.2, 12.2], [-75.2, 8.0]]]}
malla = list(h3.geo_to_cells(zona, RES))   # el poligono va en orden GeoJSON [lon, lat]
print("celdas en la malla:", len(malla))

## 4. Mapear las fuentes sobre la malla

A cada sitio se le asigna su celda H3 y, por celda, se resumen los criterios con el promedio.
Así las dos fuentes quedan unidas en una sola tabla por celda.

In [ ]:
def celda(lat, lon):
    return h3.latlng_to_cell(lat, lon, RES)

# lat/lon de la fuente A ya reproyectada
fa = fuente_a_4326.copy()
fa["lat"] = fa.geometry.y; fa["lon"] = fa.geometry.x
fa["h3"] = [celda(la, lo) for la, lo in zip(fa["lat"], fa["lon"])]
fb = fuente_b_gdf.copy()
fb["h3"] = [celda(la, lo) for la, lo in zip(fb["lat"], fb["lon"])]

agg_a = fa.groupby("h3", as_index=False).agg(sitio=("sitio", lambda x: ", ".join(x)),
                                             viento_ms=("viento_ms", "mean"),
                                             dist_via_km=("dist_via_km", "mean"))
agg_b = fb.groupby("h3", as_index=False).agg(radiacion=("radiacion", "mean"))
celdas = agg_a.merge(agg_b, on="h3", how="outer")
celdas

## 5. Visualizar

Se dibujan las celdas con dato, coloreadas por viento, para verificar que la unificación y el
mapeo quedaron bien.

In [ ]:
import branca.colormap as cm
cmap = cm.LinearColormap(["blue", "yellow", "red"],
                         vmin=celdas["viento_ms"].min(), vmax=celdas["viento_ms"].max())
cmap.caption = "Viento (m/s)"
m = folium.Map(location=[10.5, -73.0], zoom_start=7, tiles="CartoDB positron")
for _, r in celdas.iterrows():
    borde = h3.cell_to_boundary(r["h3"])
    folium.Polygon([[la, lo] for la, lo in borde], color="grey", weight=1,
                   fill=True, fill_color=cmap(r["viento_ms"]), fill_opacity=0.75,
                   tooltip=f"{r['sitio']}: viento {r['viento_ms']:.1f} m/s").add_to(m)
cmap.add_to(m)
m

## 6. Decidir: AHP y mapa de idoneidad

Con todo en la malla, se aplica AHP (Módulo 3) para combinar los criterios en un puntaje.
Criterios: viento (beneficio), distancia a vía (costo), radiación (beneficio).

In [ ]:
crit = {"viento_ms": "beneficio", "dist_via_km": "costo", "radiacion": "beneficio"}
def nz(col, sentido):
    v = celdas[col].astype(float); z = (v - v.min()) / (v.max() - v.min())
    return z if sentido == "beneficio" else 1 - z
norm = pd.DataFrame({c: nz(c, s) for c, s in crit.items()})

# pesos AHP (matriz de comparacion por pares, Modulo 3)
A = np.array([[1, 4, 2], [1/4, 1, 1/2], [1/2, 2, 1]], float)
w = (A / A.sum(0)).mean(1)
pesos = dict(zip(crit.keys(), w))
print("pesos AHP:", {c: round(v, 3) for c, v in pesos.items()})

celdas["idoneidad"] = sum(norm[c] * pesos[c] for c in crit)
celdas.sort_values("idoneidad", ascending=False)[["h3", "idoneidad"]].round(3).head()

In [ ]:
mapa = cm.LinearColormap(["blue", "yellow", "red"],
                         vmin=celdas["idoneidad"].min(), vmax=celdas["idoneidad"].max())
mapa.caption = "Idoneidad (AHP)"
m2 = folium.Map(location=[10.5, -73.0], zoom_start=7, tiles="CartoDB positron")
for _, r in celdas.iterrows():
    borde = h3.cell_to_boundary(r["h3"])
    folium.Polygon([[la, lo] for la, lo in borde], color="grey", weight=1,
                   fill=True, fill_color=mapa(r["idoneidad"]), fill_opacity=0.8,
                   tooltip=f"{r['sitio']}: idoneidad {r['idoneidad']:.2f}").add_to(m2)
mapa.add_to(m2)
m2

### Guardar el resultado

El resultado se guarda en el formato de trabajo del proyecto: `.h3.parquet` (celda H3 e idoneidad).

In [ ]:
resultado = celdas[["h3", "sitio", "viento_ms", "dist_via_km", "radiacion", "idoneidad"]].rename(
    columns={"h3": "h3_index"})
resultado.to_parquet("resultado_idoneidad.h3.parquet", index=False)
print("guardado resultado_idoneidad.h3.parquet")
resultado.sort_values("idoneidad", ascending=False).round(3).head()

## Actividad individual

Modifique los pesos de la matriz AHP, por ejemplo dándole más importancia a la cercanía a vías,
y observe cómo cambia el mapa de idoneidad.

Resultado esperado: con los pesos actuales domina el viento, y Uribia y Manaure quedan arriba
pese a estar lejos de vías. Al subir el peso de la cercanía a vías, los sitios bien conectados
como Riohacha o Barranquilla escalan posiciones y los de la alta Guajira bajan. Con los mismos
datos y otra prioridad, el resultado cambia: por eso justificar los pesos es parte de la decisión.

## Cierre

Recorrido completo del taller:

1. Fuentes en distintos CRS y formatos.
2. Unificación a un mismo sistema de coordenadas.
3. Malla H3 sobre la zona.
4. Mapeo de las fuentes a la malla.
5. Visualización.
6. Decisión con AHP y mapa de idoneidad.

Es el mismo flujo del proyecto real, a pequeña escala. Para llevarlo más lejos: más criterios,
mayor resolución en la malla, exclusiones (zonas donde no se puede) y datos reales por municipio.